# 02 — Cálculo del Índice Chocorramo

Este notebook reproduce manualmente los cálculos del índice y genera visualizaciones exploratorias.

Para generar los archivos procesados de producción, usa:
```bash
python scripts/build_index.py
```

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent
PROCESSED_DIR = ROOT / 'data' / 'processed'

with open(PROCESSED_DIR / 'purchasing_power_index.json') as f:
    index = json.load(f)

print('Índice cargado. Versión:', index['metadata']['version'])
print('Generado:', index['metadata']['generated_at'])

## Resultados principales

In [ ]:
BASE_YEAR = index['metadata']['base_year']
CURRENT_YEAR = index['metadata']['current_year']

for country_id, data in index['countries'].items():
    p = data['profile']
    calcs = data['calculations']
    base = data['snapshots'][str(BASE_YEAR)]
    curr = data['snapshots'][str(CURRENT_YEAR)]
    
    print(f"\n{p['flag']} {p['name']} — {p['product_name']}")
    print(f"  Salario mínimo {BASE_YEAR}:    {p['currency_symbol']}{base['min_wage']:,.0f} / {p['wage_period_label_es']}")
    print(f"  Salario mínimo {CURRENT_YEAR}:    {p['currency_symbol']}{curr['min_wage']:,.0f} / {p['wage_period_label_es']}")
    print(f"  Precio {p['product_name']} {BASE_YEAR}: {p['currency_symbol']}{base['product_price']:,.2f}")
    print(f"  Precio {p['product_name']} {CURRENT_YEAR}: {p['currency_symbol']}{curr['product_price']:,.2f}")
    print(f"  Comprables {BASE_YEAR}:       {base['products_purchasable']:.1f} {p['product_name']}s/{p['wage_period_label_es']}")
    print(f"  Comprables {CURRENT_YEAR}:       {curr['products_purchasable']:.1f} {p['product_name']}s/{p['wage_period_label_es']}")
    print(f"  Crecimiento salario nominal: {calcs['nominal_wage_growth_pct']:+.1f}%")
    print(f"  Crecimiento precio producto: {calcs['product_price_growth_pct']:+.1f}%")
    print(f"  Cambio poder adquisitivo:    {calcs['purchasing_power_change_pct']:+.1f}%")
    print(f"  Inflación acumulada (est.):  {calcs['accumulated_inflation_pct']:+.1f}%")
    print(f"  Cambio salario real (est.):  {calcs['real_wage_change_pct']:+.1f}%")

## Verificación de fórmulas

In [ ]:
def calc_products(wage, price):
    return wage / price

def calc_growth_pct(base, current):
    return ((current / base) - 1) * 100

def calc_real_wage(nominal, cpi_relative):
    return nominal / (cpi_relative / 100)

# Colombia verification
co = index['countries']['colombia']['snapshots']
co_base = co[str(BASE_YEAR)]
co_curr = co[str(CURRENT_YEAR)]

assert abs(calc_products(co_base['min_wage'], co_base['product_price']) - co_base['products_purchasable']) < 0.01
assert abs(calc_products(co_curr['min_wage'], co_curr['product_price']) - co_curr['products_purchasable']) < 0.01

co_calcs = index['countries']['colombia']['calculations']
assert abs(calc_growth_pct(co_base['min_wage'], co_curr['min_wage']) - co_calcs['nominal_wage_growth_pct']) < 0.1
assert abs(calc_growth_pct(co_base['product_price'], co_curr['product_price']) - co_calcs['product_price_growth_pct']) < 0.1
assert abs(calc_growth_pct(co_base['products_purchasable'], co_curr['products_purchasable']) - co_calcs['purchasing_power_change_pct']) < 0.1

print('✅ Todas las verificaciones de fórmulas pasaron.')

## Visualización — Productos comprables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (country_id, data) in zip(axes, index['countries'].items()):
    p = data['profile']
    snaps = data['snapshots']
    
    years = [str(BASE_YEAR), str(CURRENT_YEAR)]
    values = [snaps[y]['products_purchasable'] for y in years]
    colors = ['#003893' if country_id == 'colombia' else '#00008B',
              '#dc2626']
    
    bars = ax.bar(years, values, color=colors, width=0.5)
    ax.set_title(f"{p['flag']} {p['name']}\n{p['product_name']}s / {p['wage_period_label_es']}", 
                 fontsize=12, fontweight='bold')
    ax.set_ylabel(f"Unidades de {p['product_name']}")
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                f'{val:.0f}', ha='center', va='bottom', fontweight='bold')
    
    change = data['calculations']['purchasing_power_change_pct']
    color = '#16a34a' if change >= 0 else '#dc2626'
    ax.text(0.5, 0.95, f'Cambio: {change:+.1f}%', transform=ax.transAxes,
            ha='center', va='top', color=color, fontweight='bold', fontsize=11)
    
    ax.set_ylim(0, max(values) * 1.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Índice Chocorramo: Poder adquisitivo 2016 vs 2026', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'docs' / 'purchasing_power_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en docs/purchasing_power_chart.png')

## Visualización — Salario vs Precio (índice 2016=100)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (country_id, data) in zip(axes, index['countries'].items()):
    p = data['profile']
    calcs = data['calculations']
    
    years = [BASE_YEAR, CURRENT_YEAR]
    wage_index = [100, 100 + calcs['nominal_wage_growth_pct']]
    price_index = [100, 100 + calcs['product_price_growth_pct']]
    cpi_index = [100, 100 + calcs['accumulated_inflation_pct']]
    
    color_wage = '#003893' if country_id == 'colombia' else '#1e3a8a'
    
    ax.plot(years, wage_index, 'o-', color=color_wage, linewidth=2.5, 
            markersize=8, label='Salario mínimo')
    ax.plot(years, price_index, 'o-', color='#dc2626', linewidth=2.5, 
            markersize=8, label=f'Precio {p["product_name"]}')
    ax.plot(years, cpi_index, 'o--', color='#6b7280', linewidth=1.5, 
            markersize=6, label='IPC/CPI (est.)', alpha=0.7)
    ax.axhline(y=100, color='#e5e7eb', linestyle='--', linewidth=1)
    
    ax.set_title(f"{p['flag']} {p['name']} (base 2016=100)", 
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Índice (2016 = 100)')
    ax.set_xticks(years)
    ax.legend(loc='upper left', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Annotate final values
    for val, color, label in [
        (wage_index[1], color_wage, f'+{calcs["nominal_wage_growth_pct"]:.0f}%'),
        (price_index[1], '#dc2626', f'+{calcs["product_price_growth_pct"]:.0f}%'),
    ]:
        ax.annotate(label, xy=(CURRENT_YEAR, val), xytext=(5, 0), 
                   textcoords='offset points', color=color, fontweight='bold', fontsize=9)

plt.suptitle('Crecimiento nominal: Salario vs Precio del producto (2016=100)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'docs' / 'wage_vs_price_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en docs/wage_vs_price_chart.png')